# Experiment: xNES vs CMA-ES Workbench

This notebook compares the current `leitwerk.xnes.XNES` implementation against CMA-ES on the same curated objective family used by `hyperparameter_tuning.ipynb`.

- shared population size: xNES default heuristic via `_default_sample_count(None, d)`
- both optimizers start from the same mean and isotropic scale
- each function gets its own figure with clean objective at the optimizer mean vs generation
- each trial uses a random offset; `ellipsoid` and `cigar` also get a random rotation
- the transformed objective and noise stream are matched fairly across optimizers for each `function x trial` case

The goal is a fast but representative readout of optimizer state quality on the same stress family used for eta tuning.

In [ ]:
from __future__ import annotations

from collections import Counter
from collections.abc import Callable

import cma
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

from leitwerk import XNES, XNESStatus
from leitwerk.xnes import _default_sample_count

In [ ]:
# Main knobs.
DIMENSION = 30
MAX_ITERATIONS = 300
NUM_TRIALS = 3
MASTER_SEED = 1234

# Problem setup. Both optimizers see the same start state and the same transformed objective distribution.
INITIAL_SIGMA = 1.0
START_VALUE = 0.0
LOG_EPS = 1e-16
NOISE_SCALE = 1e-3
OFFSET_SCALE = 1.0
FUNCTION_NAMES = ("sphere", "ackley", "rosenbrock", "ellipsoid", "cigar")
ROTATED_FUNCTIONS = frozenset({"ellipsoid", "cigar"})

# XNES settings
XNES_UPDATE_KWARGS = {
    "eta_mu": 1.0,
    "eta_sigma": 0.5,
    "eta_B": 0.25,
}

POPULATION_SIZE = _default_sample_count(None, DIMENSION)
X0 = np.full(DIMENSION, START_VALUE, dtype=float)
TRIAL_SEEDS = MASTER_SEED + np.arange(NUM_TRIALS, dtype=int)

print(
    {
        'dimension': DIMENSION,
        'population_size': POPULATION_SIZE,
        'max_iterations': MAX_ITERATIONS,
        'num_trials': NUM_TRIALS,
        'initial_sigma': INITIAL_SIGMA,
        'noise_scale': NOISE_SCALE,
        'offset_scale': OFFSET_SCALE,
        'functions': FUNCTION_NAMES,
        'rotated_functions': sorted(ROTATED_FUNCTIONS),
    }
)

In [ ]:
def _as_matrix(points: np.ndarray) -> np.ndarray:
    x = np.asarray(points, dtype=float)
    if x.ndim == 1:
        return x[:, None]
    if x.ndim != 2:
        msg = "points must have shape (dim,) or (dim, n)."
        raise ValueError(msg)
    return x


def sphere(points: np.ndarray) -> np.ndarray:
    x = _as_matrix(points)
    return np.sum(x**2, axis=0)


def ackley(points: np.ndarray) -> np.ndarray:
    x = _as_matrix(points)
    mean_sq = np.mean(x**2, axis=0)
    mean_cos = np.mean(np.cos(2.0 * np.pi * x), axis=0)
    return -20.0 * np.exp(-0.2 * np.sqrt(mean_sq)) - np.exp(mean_cos) + 20.0 + np.e


def centered_rosenbrock(points: np.ndarray) -> np.ndarray:
    x = _as_matrix(points)
    if x.shape[0] == 1:
        return x[0] ** 2
    shifted = x + 1.0
    return np.sum(100.0 * (shifted[1:] - shifted[:-1] ** 2) ** 2 + (1.0 - shifted[:-1]) ** 2, axis=0)


def ellipsoid(points: np.ndarray) -> np.ndarray:
    x = _as_matrix(points)
    weights = 1.0e6 ** np.linspace(0.0, 1.0, x.shape[0], dtype=float)
    return np.sum(weights[:, None] * x**2, axis=0)


def cigar(points: np.ndarray) -> np.ndarray:
    x = _as_matrix(points)
    if x.shape[0] == 1:
        return x[0] ** 2
    return x[0] ** 2 + 1.0e6 * np.sum(x[1:] ** 2, axis=0)


OBJECTIVES: dict[str, Callable[[np.ndarray], np.ndarray]] = {
    "sphere": sphere,
    "ackley": ackley,
    "rosenbrock": centered_rosenbrock,
    "ellipsoid": ellipsoid,
    "cigar": cigar,
}


def random_rotation(dim: int, rng: np.random.Generator) -> np.ndarray:
    if dim <= 1:
        return np.eye(dim)
    raw = rng.standard_normal((dim, dim))
    q, r = np.linalg.qr(raw)
    signs = np.sign(np.diag(r))
    signs[signs == 0.0] = 1.0
    return q * signs


def make_case_seed(function_name: str, dim: int, trial_seed: int) -> np.random.SeedSequence:
    function_index = FUNCTION_NAMES.index(function_name)
    return np.random.SeedSequence([MASTER_SEED, function_index, dim, int(trial_seed)])


def make_noisy_objective(
    function_name: str,
    dim: int,
    trial_seed: int,
    noise_scale: float = NOISE_SCALE,
) -> tuple[Callable[[np.ndarray], float], Callable[[np.ndarray], float], dict[str, float | bool]]:
    case_seed = make_case_seed(function_name, dim, trial_seed)
    offset_seed, noise_seed, rotation_seed = case_seed.spawn(3)
    offset_rng = np.random.default_rng(offset_seed)
    noise_rng = np.random.default_rng(noise_seed)
    rotation_rng = np.random.default_rng(rotation_seed)
    offset = offset_rng.uniform(-OFFSET_SCALE, OFFSET_SCALE, size=dim)
    rotation_applied = function_name in ROTATED_FUNCTIONS and dim > 1
    rotation = random_rotation(dim, rotation_rng) if rotation_applied else np.eye(dim)
    objective = OBJECTIVES[function_name]

    def clean(x: np.ndarray) -> float:
        point = np.asarray(x, dtype=float)
        centered = point - offset
        transformed = rotation.T @ centered
        return float(objective(transformed)[0])

    def noisy(x: np.ndarray) -> float:
        base = clean(x)
        noise = noise_scale * (1.0 + base) * abs(float(noise_rng.standard_normal()))
        return float(base + noise)

    metadata: dict[str, float | bool] = {
        'rotated': rotation_applied,
        'offset_norm': float(np.linalg.norm(offset)),
    }
    return noisy, clean, metadata


def make_xnes_rng(function_name: str, dim: int, trial_seed: int) -> np.random.Generator:
    function_index = FUNCTION_NAMES.index(function_name)
    seed = np.random.SeedSequence([MASTER_SEED, function_index, dim, int(trial_seed), 1])
    return np.random.default_rng(seed)


def make_cma_seed(function_name: str, dim: int, trial_seed: int) -> int:
    function_index = FUNCTION_NAMES.index(function_name)
    seed = np.random.SeedSequence([MASTER_SEED, function_index, dim, int(trial_seed), 2])
    rng = np.random.default_rng(seed)
    return int(rng.integers(0, 2**31 - 1))

In [ ]:
def run_xnes_trial(
    objective: Callable[[np.ndarray], float],
    clean_objective: Callable[[np.ndarray], float],
    x0: np.ndarray,
    sigma0: float,
    population_size: int,
    max_iterations: int,
    rng: np.random.Generator,
) -> tuple[np.ndarray, str]:
    xnes = XNES(x0, sigma0)

    history = np.empty(max_iterations, dtype=float)
    current_value = float(clean_objective(x0))
    final_status = XNESStatus.OK.name

    for iteration in range(max_iterations):
        samples = xnes.sample_distribution(population_size, rng=rng)
        candidates = xnes.transform(samples).T
        values = np.array([float(objective(candidate)) for candidate in candidates], dtype=float)
        status = xnes.update_distribution(samples, np.argsort(values).tolist(), **XNES_UPDATE_KWARGS)
        current_value = float(clean_objective(xnes.mu))
        history[iteration] = current_value
        if status is not XNESStatus.OK:
            final_status = status.name
            history[iteration:] = current_value
            break
    else:
        return history, final_status

    return history, final_status


def run_cma_trial(
    objective: Callable[[np.ndarray], float],
    clean_objective: Callable[[np.ndarray], float],
    x0: np.ndarray,
    sigma0: float,
    population_size: int,
    max_iterations: int,
    seed: int,
) -> tuple[np.ndarray, str]:
    es = cma.CMAEvolutionStrategy(
        x0,
        sigma0,
        {
            'popsize': population_size,
            'seed': int(seed),
            'verb_disp': 0,
            'verb_log': 0,
        },
    )

    history = np.empty(max_iterations, dtype=float)
    current_value = float(clean_objective(x0))
    final_status = 'MAX_ITERATIONS'

    for iteration in range(max_iterations):
        stop_reasons = es.stop()
        if stop_reasons:
            final_status = ', '.join(sorted(str(reason) for reason in stop_reasons))
            history[iteration:] = current_value
            break

        candidates = es.ask()
        values = [float(objective(np.asarray(candidate, dtype=float))) for candidate in candidates]
        es.tell(candidates, values)
        current_value = float(clean_objective(np.asarray(es.mean, dtype=float)))
        history[iteration] = current_value
    else:
        return history, final_status

    return history, final_status


def run_suite(
    function_name: str,
    x0: np.ndarray,
    sigma0: float,
    population_size: int,
    max_iterations: int,
    trial_seeds: np.ndarray,
) -> dict[str, object]:
    curves: dict[str, list[np.ndarray]] = {'xNES': [], 'CMA-ES': []}
    statuses: dict[str, list[str]] = {'xNES': [], 'CMA-ES': []}
    problem_meta: list[dict[str, float | bool]] = []

    for trial_seed in tqdm(trial_seeds, desc=function_name):
        trial_seed = int(trial_seed)
        objective_xnes, clean_xnes, meta = make_noisy_objective(function_name, x0.size, trial_seed)
        objective_cma, clean_cma, _ = make_noisy_objective(function_name, x0.size, trial_seed)

        xnes_history, xnes_status = run_xnes_trial(
            objective_xnes,
            clean_xnes,
            x0,
            sigma0,
            population_size,
            max_iterations,
            make_xnes_rng(function_name, x0.size, trial_seed),
        )
        cma_history, cma_status = run_cma_trial(
            objective_cma,
            clean_cma,
            x0,
            sigma0,
            population_size,
            max_iterations,
            make_cma_seed(function_name, x0.size, trial_seed),
        )

        curves['xNES'].append(xnes_history)
        curves['CMA-ES'].append(cma_history)
        statuses['xNES'].append(xnes_status)
        statuses['CMA-ES'].append(cma_status)
        problem_meta.append(meta)

    return {
        'curves': {label: np.vstack(items) for label, items in curves.items()},
        'statuses': statuses,
        'problem_meta': problem_meta,
    }

In [ ]:
results_by_function = {
    function_name: run_suite(
        function_name=function_name,
        x0=X0,
        sigma0=INITIAL_SIGMA,
        population_size=POPULATION_SIZE,
        max_iterations=MAX_ITERATIONS,
        trial_seeds=TRIAL_SEEDS,
    )
    for function_name in FUNCTION_NAMES
}

for function_name, suite in results_by_function.items():
    rotated = any(bool(item['rotated']) for item in suite['problem_meta'])
    median_offset = float(np.median([float(item['offset_norm']) for item in suite['problem_meta']]))
    print(function_name, {'rotated': rotated, 'median_offset_norm': median_offset})
    for label, values in suite['curves'].items():
        print(' ', label, values.shape, dict(Counter(suite['statuses'][label])))

In [ ]:
summary: dict[str, dict[str, object]] = {}

for function_name, suite in results_by_function.items():
    function_summary: dict[str, object] = {
        'rotated': any(bool(item['rotated']) for item in suite['problem_meta']),
        'median_offset_norm': float(np.median([float(item['offset_norm']) for item in suite['problem_meta']])),
    }
    for label, values in suite['curves'].items():
        function_summary[label] = {
            'median_final_clean_mean_fitness': float(np.median(values[:, -1])),
            'best_final_clean_mean_fitness': float(np.min(values[:, -1])),
            'worst_final_clean_mean_fitness': float(np.max(values[:, -1])),
            'statuses': dict(Counter(suite['statuses'][label])),
        }
    summary[function_name] = function_summary

summary

In [ ]:
iterations = np.arange(1, MAX_ITERATIONS + 1, dtype=float)
palette = {'xNES': '#d62728', 'CMA-ES': '#1f77b4'}

for function_name in FUNCTION_NAMES:
    suite = results_by_function[function_name]
    rotated = 'rotated' if any(bool(item['rotated']) for item in suite['problem_meta']) else 'unrotated'
    median_offset = float(np.median([float(item['offset_norm']) for item in suite['problem_meta']]))

    fig, ax = plt.subplots(figsize=(8, 5))
    for label, values in suite['curves'].items():
        clipped = np.maximum(values, LOG_EPS)
        for run_index, run in enumerate(clipped):
            ax.plot(
                iterations,
                run,
                color=palette[label],
                alpha=0.5,
                linewidth=1.5,
                label=label if run_index == 0 else None,
            )

    ax.set_xscale('linear')
    ax.set_yscale('log')
    ax.set_xlabel('generation')
    ax.set_ylabel('clean fitness at mean')
    ax.set_title(
        f'{function_name} comparison, d={DIMENSION}, popsize={POPULATION_SIZE}, '
        f'trials={NUM_TRIALS}, {rotated}, median ||offset||={median_offset:.1f}'
    )
    ax.grid(True, which='both', alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()